In [1]:
import polars as pl
from tqdm import tqdm
from plotnine import *

In [2]:
filename = "norm_qced_ukb24310_c1_b3932_v1"
# filename = "norm_qced_ukb24310_c1_b8534_v1"

In [3]:
af_path = f"/home/dnanexus/data_dir/parquet_af/{filename}.parquet"

df_af = pl.read_parquet(af_path).with_columns(
    id = (pl.col("CHROM").cast(pl.Utf8) + ":" + pl.col("POS").cast(pl.Utf8) + ":" + pl.col("REF") + ":" + pl.col("ALT"))
)

df_af['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:78641620:C:T""",6219
"""chr1:78642384:C:CAAAAA""",3262
"""chr1:78643140:T:C""",3165
"""chr1:78642384:C:CAA""",2222
"""chr1:78640828:C:T""",1836
…,…
"""chr1:78643123:T:G""",1
"""chr1:78643135:A:G""",1
"""chr1:78643136:G:T""",1


In [4]:
maf_path = f"/home/dnanexus/data_dir/parquet_maf/{filename}.parquet"

df_maf = pl.read_parquet(maf_path).with_columns(
    id = (pl.col("CHROM").cast(pl.Utf8) + ":" + pl.col("POS").cast(pl.Utf8) + ":" + pl.col("REF") + ":" + pl.col("ALT"))
)

df_maf['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:78641620:C:T""",6219
"""chr1:78642384:C:CAAAAA""",3262
"""chr1:78643140:T:C""",3165
"""chr1:78642384:C:CAA""",2222
"""chr1:78640828:C:T""",1836
…,…
"""chr1:78643123:T:G""",1
"""chr1:78643135:A:G""",1
"""chr1:78643136:G:T""",1


In [16]:
tsv_path = f"/home/dnanexus/data_dir/bcf_files/output_small.tsv"
df_tsv = pl.read_csv(tsv_path, has_header=False, separator="\t").rename({
    "column_1": "CHROM",
    "column_2": "POS",
    "column_3": "REF",
    "column_4": "ALT",
    "column_5": "SAMPLE_NAME",
}).with_columns(
    id = (pl.col("CHROM").cast(pl.Utf8) + ":" + pl.col("POS").cast(pl.Utf8) + ":" + pl.col("REF") + ":" + pl.col("ALT")),
    GT = pl.col("column_6").str.split("/").list.get(0).cast(pl.Int8) + pl.col("column_6").str.split("/").list.get(1).cast(pl.Int8)
).drop(["column_6"])
df_tsv

CHROM,POS,REF,ALT,SAMPLE_NAME,id,GT
str,i64,str,str,str,str,i8
"""chr1""",78638057,"""T""","""C""","""W000001""","""chr1:78638057:T:C""",0
"""chr1""",78638057,"""T""","""C""","""W000002""","""chr1:78638057:T:C""",0
"""chr1""",78638057,"""T""","""C""","""W000003""","""chr1:78638057:T:C""",0
"""chr1""",78638057,"""T""","""C""","""W000004""","""chr1:78638057:T:C""",0
"""chr1""",78638057,"""T""","""C""","""W000005""","""chr1:78638057:T:C""",0
…,…,…,…,…,…,…
"""chr1""",78638194,"""T""","""C""","""5416178""","""chr1:78638194:T:C""",0
"""chr1""",78638194,"""T""","""C""","""2821428""","""chr1:78638194:T:C""",0
"""chr1""",78638194,"""T""","""C""","""1172302""","""chr1:78638194:T:C""",0


In [17]:
df_tsv['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:78638057:T:C""",490541
"""chr1:78638065:A:C""",490541
"""chr1:78638066:G:T""",490541
"""chr1:78638067:A:G""",490541
"""chr1:78638070:T:C""",490541
…,…
"""chr1:78638185:C:T""",490541
"""chr1:78638187:A:G""",490541
"""chr1:78638191:A:T""",490541


In [18]:
af_bcf_vars = df_af.filter(pl.col('id').is_in(df_tsv['id'].unique()))['id'].value_counts(sort=True)
af_bcf_vars

/tmp/ipykernel_114775/3429804265.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,count
str,u64
"""chr1:78638136:A:G""",65
"""chr1:78638104:G:A""",50
"""chr1:78638176:G:T""",27
"""chr1:78638073:T:G""",22
"""chr1:78638185:C:T""",11
…,…
"""chr1:78638160:G:A""",1
"""chr1:78638177:C:A""",1
"""chr1:78638183:G:T""",1


In [23]:
df_tsv.filter(pl.col('GT')>0).filter(pl.col('id').is_in(af_bcf_vars['id']))['id'].value_counts(sort=True)

/tmp/ipykernel_114775/1474594543.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,count
str,u64
"""chr1:78638136:A:G""",65
"""chr1:78638104:G:A""",50
"""chr1:78638176:G:T""",27
"""chr1:78638073:T:G""",22
"""chr1:78638185:C:T""",11
…,…
"""chr1:78638160:G:A""",1
"""chr1:78638177:C:A""",1
"""chr1:78638183:G:T""",1


In [24]:
df_maf.filter(pl.col('id').is_in(af_bcf_vars['id']))['id'].value_counts(sort=True)

/tmp/ipykernel_114775/1044796139.py:1: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.


id,count
str,u64
"""chr1:78638136:A:G""",65
"""chr1:78638104:G:A""",50
"""chr1:78638176:G:T""",27
"""chr1:78638073:T:G""",22
"""chr1:78638185:C:T""",11
…,…
"""chr1:78638160:G:A""",1
"""chr1:78638177:C:A""",1
"""chr1:78638183:G:T""",1


In [6]:
import subprocess

bcf_path = f"/home/dnanexus/data_dir/bcf_files/{filename}.bcf"

# 1. Define the fields you want using bcftools query syntax
# %CHROM, %POS, %REF, %ALT are standard
# [%SAMPLE, %GT\n] loops over samples, printing Sample Name and Genotype for every line
cmd = [
    "bcftools", "query",
    "-f", "%CHROM\t%POS\t%REF\t%ALT\t[%SAMPLE\t%GT\n]",
    bcf_path
]

# 2. Run the command and pipe output directly to Polars
# We treat the stdout as a CSV (tab-separated) file
# try:
# Get the raw output stream
process = subprocess.Popen(cmd, stdout=subprocess.PIPE)
print(process)

# Read directly into Polars
df = pl.read_csv(
    process.stdout,
    separator="\t",
    has_header=False,
    new_columns=["chrom", "pos", "ref", "alt", "sample_name", "gt"]
)

# except FileNotFoundError:
#     print("Error: bcftools not found. Please install it to use this method.")

# df

FileNotFoundError: [Errno 2] No such file or directory: 'bcftools'

In [ ]:
import pysam

bcf_path = f"/home/dnanexus/data_dir/bcf_files/{filename}.bcf"

# 1. Initialize lists to store data
data = {
    "chrom": [],
    "pos": [],
    "ref": [],
    "alts": [],
    "sample_name": [],
    "gt": []
}

with pysam.VariantFile(bcf_path) as bcf_file:
    for record in tqdm(bcf_file):
        # Pysam alts is a tuple, convert to list or join for easier reading
        alts_str = ",".join(record.alts) if record.alts else "."
        
        # Access sample genotypes
        for sample_name, sample_genotype in record.samples.items():
            # Extract basic variant info
            data["chrom"].append(record.chrom)
            data["pos"].append(record.pos)
            data["ref"].append(record.ref)
            data["alts"].append(alts_str)
            
            # Extract sample info
            data["sample_name"].append(sample_name)
            # handle case where GT might be None or missing
            gt_tuple = sample_genotype.get("GT", (None, None))
            data["gt"].append(str(gt_tuple))

# 2. Create Polars DataFrame directly from the dictionary
df = pl.DataFrame(data)

df

[E::idx_find_and_load] Could not retrieve index file for '/home/dnanexus/data_dir/bcf_files/norm_qced_ukb24310_c1_b3932_v1.bcf'
16it [00:42,  3.33s/it]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f196f4b6e10>>
Traceback (most recent call last):
  File "/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
19it [00:54,  3.75s/it]